# AncestryClassifier — Colab Training

Run preprocessing locally with Snakemake first:
```bash
snakemake --cores 4 prepare_training_data simulate_admixed
```
Then upload `data/dataset.h5` and `data/admixed_test.h5` to Google Drive and set `DRIVE_DIR` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/gene461'   # <-- change if needed

In [ ]:
import subprocess, os

# Clone repo if not already present
REPO = '/content/gene_461_final_project'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/gene_461_final_project', REPO], check=True)

%pip install -q torch h5py numpy pandas scikit-learn matplotlib

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

DATA_H5   = f'{DRIVE_DIR}/dataset.h5'
CKPT_OUT  = f'{DRIVE_DIR}/checkpoints/best_model.pt'

!python {REPO}/scripts/train.py \
    --data        {DATA_H5}  \
    --output      {CKPT_OUT} \
    --window-size 500        \
    --epochs      25         \
    --batch-size  512        \
    --num-workers 2

In [ ]:
# Optional: run evaluation on Colab too (outputs saved to Drive)
ADMIXED_H5   = f'{DRIVE_DIR}/admixed_test.h5'
CONFUSION_OUT = f'{DRIVE_DIR}/confusion_matrix.png'
KARYOGRAM_OUT = f'{DRIVE_DIR}/lai_karyogram.png'

!python {REPO}/scripts/evaluate.py \
    --data       {DATA_H5}       \
    --admixed    {ADMIXED_H5}    \
    --checkpoint {CKPT_OUT}      \
    --confusion  {CONFUSION_OUT} \
    --karyogram  {KARYOGRAM_OUT} \
    --window-size 500

In [ ]:
from IPython.display import Image
Image(CONFUSION_OUT)

In [ ]:
Image(KARYOGRAM_OUT)